# Inherent Structures: Langevin to Quench
This notebook runs the temperature experiment directly from Colab: equilibrate with full-batch Langevin dynamics, quench with full-batch gradient descent, and measure the loss and overlap at the inherent structure.

In [ ]:
# Run this cell only when starting from a fresh Colab runtime.
# Replace REPO_URL with the repository URL or mount the project from Drive.
REPO_URL = "https://github.com/marjankowskam/Comparing_Dynamics.git"
!git clone {REPO_URL}
%cd Comparing_Dynamics
%pip install -r requirements.txt
%pip install -e .

In [ ]:
from pathlib import Path
import torch
import wandb

from comparing_dynamics.config import Config
from comparing_dynamics.experiments.inherent_structures import (
    InherentStructureConfig,
    run_inherent_structure,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
output_dir = Path("runs/inherent-structures")

In [ ]:
cfg = Config(
    initial_seed=19,
    task_name="fashion_mnist",
    task_params={"batch_size": None, "data_dir": "data"},
    model_name="mnistMLP",
    model_params={
        "hidden_dim": [100, 100, 10],
        "activation_fn": "relu",
    },
)

equilibration_steps = 1000
quench_steps = 5000
langevin_lr = 1e-3
quench_lr = 1e-2
temperatures = [0.0, 0.01, 0.1, 1.0]

In [ ]:
results = []

for temperature in temperatures:
    cfg.initial_seed = 19
    wb_run = wandb.init(
        project="comparing-dynamics",
        name=f"fashion-mnist-T-{temperature:g}",
        config={
            "temperature": temperature,
            "equilibration_steps": equilibration_steps,
            "quench_steps": quench_steps,
            "langevin_lr": langevin_lr,
            "quench_lr": quench_lr,
            "device": device,
        },
    )
    try:
        experiment_cfg = InherentStructureConfig(
            temperature=temperature,
            equilibration_steps=equilibration_steps,
            quench_steps=quench_steps,
            langevin_lr=langevin_lr,
            quench_lr=quench_lr,
            seed=cfg.initial_seed,
        )
        result = run_inherent_structure(
            cfg,
            experiment_cfg,
            device=device,
            save_dir=output_dir,
            run_name=wb_run.name,
        )
        wandb.log({
            "inherent_loss": result["final_loss"],
            "inherent_gradient_norm": result["final_gradient_norm"],
            "overlap_q": result["overlap_q"],
            "initial_to_inherent_overlap": result["initial_to_inherent_overlap"],
        })
        results.append(result)
    finally:
        wandb.finish()

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        "temperature": result["temperature"],
        "inherent_loss": result["final_loss"],
        "gradient_norm": result["final_gradient_norm"],
        "overlap_q": result["overlap_q"],
        "initial_to_inherent_overlap": result["initial_to_inherent_overlap"],
    }
    for result in results
])
summary

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(summary["temperature"], summary["inherent_loss"], marker="o")
axes[0].set(xlabel="Temperature T", ylabel="Inherent-structure loss")
axes[1].plot(summary["temperature"], summary["overlap_q"], marker="o")
axes[1].set(xlabel="Temperature T", ylabel="q(T)")
fig.tight_layout()

# SGD to Quench

In [1]:
import torch 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl 

In [ ]:
# Paths to the exported run metadata and saved SGD artifacts.
base_path = Path("G:/My Drive/STUDIA/PhD/ICML_2026_Hessian_Eigenvectors/")
runs_path = Path("../data/Comparing_Dynamics.xlsx")
metrics_path = Path("../data/Comparing_Dynamics_wandb_metrics.xlsx")

# Load SGD runs and optional metrics exported from Weights & Biases.
df = pd.read_excel(runs_path, sheet_name="ICML_runs")
df_metrics = (
    pd.read_excel(metrics_path, sheet_name="ICML_runs")
    if metrics_path.exists()
    else pd.DataFrame()
)
filtered = df[
    df["op_name"].str.lower().eq("sgd")
    & df["task_name"].str.lower().eq("mnist")
].drop_duplicates(subset=["run_name"])

quench_steps = 5000
quench_lr = 1e-2
quench_tolerance = 0.0
sgd_to_quench_dir = output_dir / "sgd-to-quench"
sgd_to_quench_results = []

for run_name in filtered["run_name"]:
    try:
        run_cfg = Config.load_from_file(run_name, str(base_path) + "/")
        checkpoint = torch.load(
            base_path / f"Model_{run_name}.pt",
            map_location=device,
            weights_only=False,
        )
    except FileNotFoundError:
        print(f"Missing config or checkpoint: {run_name}")
        continue

    model = get_model(run_cfg.model_name, run_cfg.model_params)
    model.load_state_dict(checkpoint["model_state_dict"])
    experiment_cfg = InherentStructureConfig(
        temperature=0.0,
        equilibration_steps=0,
        quench_steps=quench_steps,
        langevin_lr=0.0,
        quench_lr=quench_lr,
        quench_tolerance=quench_tolerance,
        seed=run_cfg.initial_seed,
    )
    result = run_inherent_structure(
        run_cfg,
        experiment_cfg,
        device=device,
        model=model,
        save_dir=sgd_to_quench_dir,
        run_name=run_name,
    )
    result["run_name"] = run_name
    result["sgd_metrics"] = (
        df_metrics[df_metrics["run_name"].eq(run_name)].iloc[0].to_dict()
        if not df_metrics.empty and (df_metrics["run_name"] == run_name).any()
        else {}
    )
    sgd_to_quench_results.append(result)

sgd_to_quench_summary = pd.DataFrame([
    {
        "run_name": result["run_name"],
        "inherent_loss": result["final_loss"],
        "gradient_norm": result["final_gradient_norm"],
        "sgd_to_inherent_overlap": result["overlap_q"],
        "checkpoint": result["saved_files"]["inherent_checkpoint"],
    }
    for result in sgd_to_quench_results
])
sgd_to_quench_summary

SyntaxError: invalid syntax (784069727.py, line 11)